# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Spatial Domain Image Enhancement and Digital Filtering
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/03_SpatialDomainFiltering.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
This interactive Jupyter Notebook presents the theoretical foundations, signal processing mechanics, and practical implementations of **Spatial Domain Image Enhancement and Digital Filtering**.

By completing this notebook, you will learn to:
1. Apply **point processing intensity transformations**: Logarithmic dynamic range compression and Power-Law (Gamma) contrast correction.
2. Implement **piecewise-linear transformations** and **intensity-level slicing** (binary vs. background-preserving).
3. Perform **bit-plane decomposition**, evaluate structural contributions of MSB vs. LSB, and analyze reconstruction error via **residual maps**.
4. Compute and interpret discrete intensity **histograms**, Probability Density Functions (PDF), Cumulative Distribution Functions (CDF), and statistical moments.
5. Perform **Global Histogram Equalization** and **Histogram Matching (Specification)**.
6. Contrast **Global vs. Local (Block-based) Histogram Equalization** for revealing fine local details in medical modalities.
7. Understand 2D **spatial convolution mechanics** step-by-step through interactive matrix visualizations.
8. Apply **spatial low-pass smoothing** (Box and Gaussian filters), and evaluate non-linear **Median Filtering** for impulse (salt-and-pepper) noise reduction.
9. Implement **high-pass spatial filters**, gradient operators (**Sobel**), isotropic second-order derivatives (**Laplacian**), and **Unsharp Masking / High-Boost Filtering**.
10. Explore **advanced spatial enhancement techniques**: Brightness-Preserving Bi-Histogram Equalization (BBHE), Contrast Limited Adaptive Histogram Equalization (CLAHE), Frangi Multiscale Hessian Vesselness Filtering, Fuzzy Logic Contrast Enhancement, Total Variation (TV) Denoising, and Perona–Malik Anisotropic Diffusion.

---


---
## SECTION 0: Google Colab Environment Setup & Asset Verification

### Theoretical & Engineering Setup
To guarantee seamless execution both locally and inside Google Colab, this module creates the `./lab_materials/` workspace directory, verifies local dataset image assets (`cameraman.tif`, `ct_thorax.png`, `mri_t1n_brain.png`), automatically fetches any missing test targets, and sets high-resolution Matplotlib inline plotting parameters.


In [ ]:
# Section 0: Google Colab Environment Setup & Asset Verification
import os
import sys
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage, signal
from PIL import Image

# Automated package installation for Colab
try:
    import skimage
    import skimage.filters
    import skimage.restoration
    import skimage.exposure
except ImportError:
    print("Installing scikit-image...")
    os.system(f"{sys.executable} -m pip install -q scikit-image")
    import skimage
    import skimage.filters
    import skimage.restoration
    import skimage.exposure

# 1. Ensure target directory structure exists
lab_dir = "./example_data"
os.makedirs(lab_dir, exist_ok=True)

# 2. Automated download of required image assets if missing
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Astarakee/isa-su/main/labs/example_data"
ASSETS = {
    "cameraman.tif": f"{GITHUB_RAW_BASE}/cameraman.tif",
    "ct_thorax.png": f"{GITHUB_RAW_BASE}/ct_thorax.png",
    "mri_t1n_brain.png": f"{GITHUB_RAW_BASE}/mri_t1n_brain.png"
}

for fname, url in ASSETS.items():
    fpath = os.path.join(lab_dir, fname)
    if not os.path.exists(fpath):
        print(f"Downloading missing asset '{fname}' into {lab_dir}...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as resp, open(fpath, 'wb') as f:
                f.write(resp.read())
            print(f"Successfully downloaded '{fname}'")
        except Exception as e:
            print(f"Warning: Failed to download '{fname}': {e}")

# Verify clean image format for cameraman if downloaded as TIFF
cameraman_path = os.path.join(lab_dir, "cameraman.tif")
if os.path.exists(cameraman_path):
    try:
        pil_cam = Image.open(cameraman_path).convert('L')
        pil_cam.save(cameraman_path)
    except Exception as e:
        print(f"TIFF cleanup notice: {e}")

# 3. Configure high-quality inline plotting defaults
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = False
plt.rcParams['figure.autolayout'] = True

print("Environment setup complete. All required libraries and assets are ready.")

---
## SECTION 1: Intensity Transformations (Logarithmic and Power-Law / Gamma)

### Theoretical Background
Spatial domain point processing operates directly on individual pixel intensities independent of spatial coordinates:
$$s = T(r)$$
where $r \in [0, L-1]$ is the input intensity and $s \in [0, L-1]$ is the processed output intensity.

#### 1. Logarithmic Transformation
$$s = c \cdot \ln(1 + r)$$
where $c = \frac{L-1}{\ln(1 + r_{\max})}$ is a normalization scaling constant. Logarithmic transformation expands narrow ranges of low-intensity dark values while compressing high-intensity bright values. It is widely applied when displaying dynamic ranges spanning several orders of magnitude (e.g., Fourier transform magnitude spectra).

#### 2. Power-Law (Gamma $\gamma$) Transformation
$$s = c \cdot r^\gamma$$
where $c$ is a normalization constant (assuming $r \in [0, 1]$).
- **Fractional Gamma ($\gamma < 1$)**: Expands dark region contrast, brightening shadow details.
- **Steep Gamma ($\gamma > 1$)**: Expands bright region contrast, compressing dark background intensities.


In [ ]:
# Section 1: Code Implementation

# Load Cameraman Image
cam_path = os.path.join(lab_dir, "cameraman.tif")
img_cam = cv2.imread(cam_path, cv2.IMREAD_GRAYSCALE)

if img_cam is None:
    print("Warning: cameraman.tif missing. Generating synthetic test target.")
    img_cam = (np.outer(np.linspace(10, 240, 256), np.linspace(10, 240, 256))).astype(np.uint8)

# Normalize intensity to float [0, 1]
r_norm = img_cam.astype(np.float32) / 255.0

# 1. Logarithmic Transformation
c_log = 1.0 / np.log(1.0 + 1.0)
s_log = c_log * np.log(1.0 + r_norm)
img_log = (s_log * 255.0).clip(0, 255).astype(np.uint8)

# 2. Power-Law (Gamma) Corrections
gammas = [0.4, 0.7, 1.5, 2.5]
gamma_images = []

for g in gammas:
    s_gamma = np.power(r_norm, g)
    img_g = (s_gamma * 255.0).clip(0, 255).astype(np.uint8)
    gamma_images.append(img_g)

# Display Results
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

axes[0, 0].imshow(img_cam, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title("Original Image r")
axes[0, 0].axis('off')

axes[0, 1].imshow(img_log, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title("Log Transformation s = c*ln(1+r)")
axes[0, 1].axis('off')

axes[0, 2].imshow(gamma_images[0], cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title(f"Gamma Correction (gamma = {gammas[0]})")
axes[0, 2].axis('off')

axes[1, 0].imshow(gamma_images[1], cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title(f"Gamma Correction (gamma = {gammas[1]})")
axes[1, 0].axis('off')

axes[1, 1].imshow(gamma_images[2], cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title(f"Gamma Correction (gamma = {gammas[2]})")
axes[1, 1].axis('off')

axes[1, 2].imshow(gamma_images[3], cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title(f"Gamma Correction (gamma = {gammas[3]})")
axes[1, 2].axis('off')

plt.suptitle("Intensity Point Transformations: Logarithmic vs. Power-Law (Gamma)", fontsize=14)
plt.tight_layout()
plt.show()


---
## SECTION 2: Piecewise-Linear Transformations & Intensity-Level Slicing

### Theoretical Background
Piecewise-linear transformation functions allow arbitrary non-linear mapping curves $s = T(r)$ composed of connected linear segments:

#### 1. Contrast Stretching
Stretches a narrow input intensity dynamic range $[r_{\min}, r_{\max}]$ to cover the full display range $[0, L-1]$:
$$s = \begin{cases}
\frac{s_1}{r_1} r, & 0 \le r < r_1 \\
\frac{s_2 - s_1}{r_2 - r_1}(r - r_1) + s_1, & r_1 \le r < r_2 \\
\frac{(L-1) - s_2}{(L-1) - r_2}(r - r_2) + s_2, & r_2 \le r \le L-1
\end{cases}$$

#### 2. Intensity-Level Slicing
Isolates a specific intensity band $[A, B]$ of interest (e.g., highlighting specific tissue densities in medical imaging):
- **Scheme A (Binary Output / Without Background)**:
  $$s = \begin{cases} L-1, & A \le r \le B \\ 0, & \text{otherwise} \end{cases}$$
- **Scheme B (Preserving Background)**:
  $$s = \begin{cases} L-1, & A \le r \le B \\ r, & \text{otherwise} \end{cases}$$


In [ ]:
# Section 2: Code Implementation

# Load Cameraman Image
r_img = img_cam.copy()
L = 256

# 1. Piecewise Linear Contrast Stretching Function
r1, s1 = 70, 20
r2, s2 = 180, 230

def contrast_stretch(r):
    s = np.zeros_like(r, dtype=np.float32)
    mask1 = (r < r1)
    mask2 = (r >= r1) & (r < r2)
    mask3 = (r >= r2)
    
    s[mask1] = (s1 / r1) * r[mask1]
    s[mask2] = ((s2 - s1) / (r2 - r1)) * (r[mask2] - r1) + s1
    s[mask3] = ((L - 1 - s2) / (L - 1 - r2)) * (r[mask3] - r2) + s2
    return np.clip(s, 0, 255).astype(np.uint8)

img_cs = contrast_stretch(r_img)

# 2. Intensity-Level Slicing (Range A=100 to B=180)
A, B = 100, 180

# Scheme A: Binary output (No background)
img_slice_bin = np.where((r_img >= A) & (r_img <= B), 255, 0).astype(np.uint8)

# Scheme B: Background preserved
img_slice_bg = np.where((r_img >= A) & (r_img <= B), 255, r_img).astype(np.uint8)

# Plotting Transformation Curves and Processed Images
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Curve 1: Contrast Stretching
r_vals = np.arange(256)
axes[0, 0].plot(r_vals, contrast_stretch(r_vals), 'b-', lw=2)
axes[0, 0].plot([r1, r2], [s1, s2], 'ro', markersize=6)
axes[0, 0].set_title("Curve: Contrast Stretching")
axes[0, 0].set_xlabel("Input Intensity r")
axes[0, 0].set_ylabel("Output Intensity s")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].imshow(r_img, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title("Original Image")
axes[0, 1].axis('off')

axes[0, 2].imshow(img_cs, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title("Contrast Stretched Image")
axes[0, 2].axis('off')

# Curve 2: Intensity-Level Slicing
curve_bin = np.where((r_vals >= A) & (r_vals <= B), 255, 0)
curve_bg = np.where((r_vals >= A) & (r_vals <= B), 255, r_vals)

axes[1, 0].plot(r_vals, curve_bin, 'r--', label='Binary Slicing', lw=2)
axes[1, 0].plot(r_vals, curve_bg, 'g-', label='Preserving Background', lw=2)
axes[1, 0].set_title(f"Curves: Intensity Slicing [{A}, {B}]")
axes[1, 0].set_xlabel("Input Intensity r")
axes[1, 0].set_ylabel("Output Intensity s")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].imshow(img_slice_bin, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title("Slicing (Binary Output)")
axes[1, 1].axis('off')

axes[1, 2].imshow(img_slice_bg, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title("Slicing (Background Preserved)")
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


---
## SECTION 3: Bit-Plane Decomposition, Reconstruction, and Residual Analysis

### Theoretical Background
An 8-bit digital grayscale image $I(x, y) \in [0, 255]$ can be decomposed into eight independent binary bit planes $b_0(x,y), b_1(x,y), \dots, b_7(x,y) \in \{0, 1\}$:

$$I(x, y) = \sum_{k=0}^{7} b_k(x, y) \cdot 2^k$$

#### Cumulative Reconstruction using MSBs
Reconstructing an image using only its top $M$ Most Significant Bits (MSBs) sets lower bits to zero:
$$I_{\text{rec, } M}(x, y) = \sum_{k=8-M}^{7} b_k(x, y) \cdot 2^k$$

#### Residual Reconstruction Error
The information loss between the original image and a reduced bit-depth reconstructed image is captured by the spatial residual map:
$$R_M(x, y) = \big| I(x, y) - I_{\text{rec, } M}(x, y) \big|$$


In [ ]:
# Section 3: Code Implementation

# Load CT Thorax Image
ct_path = os.path.join(lab_dir, "ct_thorax.png")
img_ct = cv2.imread(ct_path, cv2.IMREAD_GRAYSCALE)

if img_ct is None:
    print("Warning: ct_thorax.png missing. Generating synthetic thorax phantom.")
    h, w = 300, 300
    y, x = np.ogrid[:h, :w]
    mask = ((x-150)**2 + (y-150)**2) <= 100**2
    img_ct = (mask * 180 + np.random.randint(0, 20, (h, w))).astype(np.uint8)

# 1. Extract All 8 Bit Planes
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

bit_planes_ct = []
for k in range(8):
    b_k = (img_ct >> k) & 1
    bit_planes_ct.append(b_k)
    axes[k].imshow(b_k, cmap='gray')
    axes[k].set_title(f"Bit Plane {k} (2^{k} = {2**k})")
    axes[k].axis('off')

plt.suptitle("CT Thorax Bit-Plane Decomposition (Bit 0 LSB to Bit 7 MSB)", fontsize=13)
plt.tight_layout()
plt.show()

# 2. Cumulative MSB Reconstruction
# Rec 1: Bits {6, 7} (Top 2 MSBs)
rec_top2 = (bit_planes_ct[7] * 128) + (bit_planes_ct[6] * 64)

# Rec 2: Bits {5, 6, 7} (Top 3 MSBs)
rec_top3 = rec_top2 + (bit_planes_ct[5] * 32)

# Rec 3: Bits {4, 5, 6, 7} (Top 4 MSBs)
rec_top4 = rec_top3 + (bit_planes_ct[4] * 16)

reconstructions = [rec_top2, rec_top3, rec_top4]
rec_titles = ["Top 2 MSBs {6,7}", "Top 3 MSBs {5,6,7}", "Top 4 MSBs {4,5,6,7}"]

# 3. Residual Error Maps
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for idx, (rec_img, title) in enumerate(zip(reconstructions, rec_titles)):
    # Display Reconstructed Image
    axes[0, idx].imshow(rec_img, cmap='gray', vmin=0, vmax=255)
    axes[0, idx].set_title(f"Reconstructed: {title}")
    axes[0, idx].axis('off')
    
    # Calculate Residual
    residual = np.abs(img_ct.astype(np.float32) - rec_img.astype(np.float32))
    mae = np.mean(residual)
    
    im = axes[1, idx].imshow(residual, cmap='hot', vmin=0, vmax=64)
    axes[1, idx].set_title(f"Residual Error |Orig - Rec|\n(MAE: {mae:.2f})")
    axes[1, idx].axis('off')
    fig.colorbar(im, ax=axes[1, idx], fraction=0.046, pad=0.04)

plt.suptitle("MSB Reconstruction and Residual Analysis on CT Thorax", fontsize=14)
plt.tight_layout()
plt.show()


---
## SECTION 4: Image Histograms and Statistical Analysis

### Theoretical Background
The intensity histogram of a 2D digital image $I(x, y)$ of size $M \times N$ with $L$ intensity levels ($0$ to $L-1$) is a discrete function:
$$h(r_k) = n_k, \quad k = 0, 1, \dots, L-1$$
where $r_k$ is the $k$-th intensity value and $n_k$ is the number of pixels having intensity $r_k$.

#### Normalized Probability Density Function (PDF)
$$p(r_k) = \frac{n_k}{M \cdot N}, \quad \text{such that } \sum_{k=0}^{L-1} p(r_k) = 1$$

#### Statistical Moments
- **Mean Intensity (Average Brightness)**:
  $$\mu = \sum_{k=0}^{L-1} r_k \cdot p(r_k)$$
- **Variance (Global Contrast Measure)**:
  $$\sigma^2 = \sum_{k=0}^{L-1} (r_k - \mu)^2 \cdot p(r_k)$$


In [ ]:
# Section 4: Code Implementation

# Load Cameraman Image
img_target = img_cam.copy()
M, N = img_target.shape

# 1. Compute Discrete Histogram using NumPy
hist_counts, bin_edges = np.histogram(img_target, bins=256, range=(0, 256))
pdf = hist_counts / (M * N)

# 2. Compute Statistical Parameters
intensity_levels = np.arange(256)
mean_val = np.sum(intensity_levels * pdf)
var_val = np.sum(((intensity_levels - mean_val)**2) * pdf)
std_val = np.sqrt(var_val)

print(f"Cameraman Histogram Statistics:")
print(f" - Image Size:          {M} x {N} pixels ({M*N} total)")
print(f" - Mean Intensity (mu): {mean_val:.2f}")
print(f" - Variance (sigma^2):   {var_val:.2f}")
print(f" - Std Deviation (sigma):{std_val:.2f}")

# 3. Visualize Image, Histogram, and PDF
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(img_target, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Cameraman Grayscale Image")
axes[0].axis('off')

# Plot Absolute Counts Histogram
axes[1].bar(intensity_levels, hist_counts, width=1.0, color='gray', edgecolor='black', alpha=0.7)
axes[1].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f"Mean (mu = {mean_val:.1f})")
axes[1].set_title("Intensity Histogram h(r_k)")
axes[1].set_xlabel("Intensity Level r_k")
axes[1].set_ylabel("Pixel Count n_k")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot Normalized Probability Density Function (PDF)
axes[2].bar(intensity_levels, pdf, width=1.0, color='navy', alpha=0.7)
axes[2].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f"Mean (mu = {mean_val:.1f})")
axes[2].set_title("Normalized PDF p(r_k)")
axes[2].set_xlabel("Intensity Level r_k")
axes[2].set_ylabel("Probability p(r_k)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## SECTION 5: Global Histogram Equalization

### Theoretical Background
Histogram Equalization transforms the intensity distribution of an image to approximate a uniform distribution, maximizing global image entropy and contrast.

#### Cumulative Distribution Function (CDF) Mapping
The monotonic transformation function $s = T(r)$ is derived directly from the normalized CDF:
$$s_k = T(r_k) = (L-1) \sum_{j=0}^{k} p(r_j) = (L-1) \cdot \text{CDF}(r_k)$$
where $L=256$ for 8-bit images. Each input intensity $r_k$ is remapped to output intensity $s_k = \text{round}(T(r_k))$.


In [ ]:
# Section 5: Code Implementation

# 1. Perform Global Histogram Equalization
img_eq = cv2.equalizeHist(img_cam)

# Compute Histograms and CDFs for Original and Equalized Images
hist_orig, _ = np.histogram(img_cam, bins=256, range=(0, 256))
pdf_orig = hist_orig / img_cam.size
cdf_orig = np.cumsum(pdf_orig)

hist_eq, _ = np.histogram(img_eq, bins=256, range=(0, 256))
pdf_eq = hist_eq / img_eq.size
cdf_eq = np.cumsum(pdf_eq)

# 2. Visualize Original vs. Equalized with Histograms & CDF Overlays
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Original Image & Histogram
axes[0, 0].imshow(img_cam, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title("Original Image")
axes[0, 0].axis('off')

ax_hist0 = axes[0, 1]
ax_cdf0 = ax_hist0.twinx()

ax_hist0.bar(np.arange(256), pdf_orig, color='gray', alpha=0.6, label='Histogram (PDF)')
ax_cdf0.plot(np.arange(256), cdf_orig, color='blue', lw=2, label='CDF T(r)')
ax_hist0.set_title("Original Histogram & CDF")
ax_hist0.set_xlabel("Intensity Level r")
ax_hist0.set_ylabel("Probability PDF", color='gray')
ax_cdf0.set_ylabel("Cumulative CDF", color='blue')
ax_hist0.grid(True, alpha=0.3)

# Equalized Image & Histogram
axes[1, 0].imshow(img_eq, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title("Global Histogram Equalized Image")
axes[1, 0].axis('off')

ax_hist1 = axes[1, 1]
ax_cdf1 = ax_hist1.twinx()

ax_hist1.bar(np.arange(256), pdf_eq, color='gray', alpha=0.6, label='Histogram (PDF)')
ax_cdf1.plot(np.arange(256), cdf_eq, color='red', lw=2, label='Equalized CDF')
ax_hist1.set_title("Equalized Histogram & Linearized CDF")
ax_hist1.set_xlabel("Intensity Level s")
ax_hist1.set_ylabel("Probability PDF", color='gray')
ax_cdf1.set_ylabel("Cumulative CDF", color='red')
ax_hist1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## SECTION 6: Histogram Matching (Specification)

### Theoretical Background
Histogram Matching (or Histogram Specification) generates an output image whose intensity distribution matches a specified target reference histogram $p_Z(z)$.

#### Mathematical Algorithm
1. **Equalize Source Image $X$**: Compute transformation $s = T(r) = (L-1) \sum_{j=0}^{r} p_X(r_j)$.
2. **Equalize Target Image $Z$**: Compute transformation $v = G(z) = (L-1) \sum_{k=0}^{z} p_Z(z_k)$.
3. **Inverse Mapping**: Find $z = G^{-1}(s) = G^{-1}(T(r))$ by matching equalized intensity values $s \approx v$.


In [ ]:
# Section 6: Code Implementation

# Load Source (Cameraman) and Target Reference (MRI Brain)
img_source = img_cam.copy()

mri_path = os.path.join(lab_dir, "mri_t1n_brain.png")
img_target_ref = cv2.imread(mri_path, cv2.IMREAD_GRAYSCALE)

if img_target_ref is None:
    print("Warning: mri_t1n_brain.png missing. Generating synthetic reference target.")
    img_target_ref = (np.linspace(20, 220, 256*256).reshape(256, 256)).astype(np.uint8)

# Custom Fallback for Histogram Matching
def match_histograms_custom(source, reference):
    oldshape = source.shape
    src = source.ravel()
    ref = reference.ravel()
    s_values, bin_idx, s_counts = np.unique(src, return_inverse=True, return_counts=True)
    r_values, r_counts = np.unique(ref, return_counts=True)
    s_quantiles = np.cumsum(s_counts) / src.size
    r_quantiles = np.cumsum(r_counts) / ref.size
    interp_t_values = np.interp(s_quantiles, r_quantiles, r_values)
    return interp_t_values[bin_idx].reshape(oldshape).astype(source.dtype)

try:
    img_matched = skimage.exposure.match_histograms(img_source, img_target_ref).astype(np.uint8)
except Exception:
    img_matched = match_histograms_custom(img_source, img_target_ref)

# Compute Histograms
pdf_source, _ = np.histogram(img_source, bins=256, range=(0, 256), density=True)
pdf_ref, _ = np.histogram(img_target_ref, bins=256, range=(0, 256), density=True)
pdf_matched, _ = np.histogram(img_matched, bins=256, range=(0, 256), density=True)

# Display Comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].imshow(img_source, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title("Source Image (Cameraman)")
axes[0, 0].axis('off')

axes[0, 1].imshow(img_target_ref, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title("Target Reference (Brain MRI)")
axes[0, 1].axis('off')

axes[0, 2].imshow(img_matched, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title("Matched Output Image")
axes[0, 2].axis('off')

# Plot Histograms
axes[1, 0].plot(pdf_source, color='blue', lw=2)
axes[1, 0].set_title("Source PDF")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(pdf_ref, color='green', lw=2)
axes[1, 1].set_title("Target Reference PDF")
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(pdf_matched, color='red', lw=2)
axes[1, 2].set_title("Matched Output PDF")
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle("Histogram Matching (Specification)", fontsize=14)
plt.tight_layout()
plt.show()


---
## SECTION 7: Local vs. Global Histogram Equalization

### Theoretical Background
- **Global Histogram Equalization**: Uses the intensity distribution across the entire image. Highly efficient, but can wash out local details or over-amplify noise in low-contrast homogeneous sub-regions.
- **Local (Adaptive) Histogram Equalization**: Operates over sliding local sub-windows / blocks ($N_w \times N_w$). Local CDFs are computed independently per neighborhood, preserving fine anatomical boundary details in medical images.


In [ ]:
# Section 7: Code Implementation

# Load Brain MRI
img_mri = img_target_ref.copy()

# 1. Global Histogram Equalization
img_mri_global = cv2.equalizeHist(img_mri)

# 2. Local Contrast Limited Adaptive Histogram Equalization (CLAHE)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
img_mri_local = clahe.apply(img_mri)

# Display Comparison with Inset Zoom Highlights
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))

# Define Zoom Region Coordinates (e.g. brain tissue detail)
h_m, w_m = img_mri.shape
y1, y2 = int(h_m*0.35), int(h_m*0.65)
x1, x2 = int(w_m*0.35), int(w_m*0.65)

# Original
axes[0].imshow(img_mri, cmap='gray')
axes[0].set_title("Original Brain MRI")
rect0 = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='yellow', lw=2)
axes[0].add_patch(rect0)
axes[0].axis('off')

# Global Equalization
axes[1].imshow(img_mri_global, cmap='gray')
axes[1].set_title("Global Histogram Equalization")
rect1 = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='yellow', lw=2)
axes[1].add_patch(rect1)
axes[1].axis('off')

# Local CLAHE Equalization
axes[2].imshow(img_mri_local, cmap='gray')
axes[2].set_title("Local Equalization (CLAHE 8x8)")
rect2 = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='yellow', lw=2)
axes[2].add_patch(rect2)
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Zoomed-In Subplot Row
fig, axes_zoom = plt.subplots(1, 3, figsize=(15, 4))

axes_zoom[0].imshow(img_mri[y1:y2, x1:x2], cmap='gray')
axes_zoom[0].set_title("Zoomed Region (Original)")
axes_zoom[0].axis('off')

axes_zoom[1].imshow(img_mri_global[y1:y2, x1:x2], cmap='gray')
axes_zoom[1].set_title("Zoomed Region (Global Eq)")
axes_zoom[1].axis('off')

axes_zoom[2].imshow(img_mri_local[y1:y2, x1:x2], cmap='gray')
axes_zoom[2].set_title("Zoomed Region (Local CLAHE)")
axes_zoom[2].axis('off')

plt.tight_layout()
plt.show()


---
## SECTION 8: Spatial Filtering Fundamentals & Step-by-Step Convolution Mechanics

### Theoretical Background
Spatial domain linear filtering computes a weighted sum of pixel intensities within a local spatial neighborhood:

$$g(x, y) = w(x, y) * f(x, y) = \sum_{s=-a}^{a} \sum_{t=-b}^{b} w(s, t) \cdot f(x - s, y - t)$$

where $w(s, t)$ is a spatial filter mask (kernel) of dimensions $(2a+1) \times (2b+1)$.
- **Boundary Handling**: Zero-padding, Replicate/Edge-padding, Reflect-padding.


In [ ]:
# Section 8: Code Implementation

# Synthesize a 31x31 Discrete Target Pattern
grid_size = 31
synth_img = np.zeros((grid_size, grid_size), dtype=np.float32)
synth_img[10:21, 10:21] = 100.0
synth_img[14:17, 14:17] = 200.0

# Define a 3x3 Averaging Kernel
kernel_3x3 = np.ones((3, 3), dtype=np.float32) / 9.0

# Coordinates to visualize step-by-step sliding convolution
coords = [(5, 5), (15, 15), (25, 25)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (cy, cx) in enumerate(coords):
    # Extract 3x3 local neighborhood
    neighborhood = synth_img[cy-1:cy+2, cx-1:cx+2]
    elem_mult = neighborhood * kernel_3x3
    conv_val = np.sum(elem_mult)
    
    # Plot Image with Overlay Kernel Box
    axes[idx].imshow(synth_img, cmap='viridis', origin='upper')
    rect = plt.Rectangle((cx-1.5, cy-1.5), 3, 3, fill=False, color='red', lw=2.5)
    axes[idx].add_patch(rect)
    axes[idx].set_title(f"Position ({cx},{cy}) -> Conv Output = {conv_val:.2f}")
    axes[idx].set_xlabel("Column Index x")
    axes[idx].set_ylabel("Row Index y")

plt.suptitle("Step-by-Step 2D Spatial Convolution Mechanics", fontsize=14)
plt.tight_layout()
plt.show()


---
## SECTION 9: Spatial Low-Pass Smoothing and Noise Suppression

### Theoretical Background
- **Box Averaging Filter**: Uniform kernel $w = \frac{1}{K^2} \mathbf{1}_{K \times K}$. Smooths image, but introduces step-discontinuity ringing artifacts.
- **Gaussian Low-Pass Filter**: Isotropic Gaussian kernel $w(x,y) = \frac{1}{2\pi \sigma^2} \exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)$. Smooth, artifact-free blurring.
- **Median Filter**: Non-linear rank-order filter $g(x,y) = \text{median}\{f(x+s,y+t)\}$. Highly effective at removing impulse (salt-and-pepper) noise while preserving sharp boundaries.


In [ ]:
# Section 9: Code Implementation

# Load CT Thorax
img_smooth_base = img_ct.copy()

# 1. Box Filtering Across Multiple Kernel Sizes
box_sizes = [3, 5, 7, 9, 11, 19, 27]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

axes[0].imshow(img_smooth_base, cmap='gray')
axes[0].set_title("Original CT Thorax")
axes[0].axis('off')

for idx, ksize in enumerate(box_sizes):
    box_filtered = cv2.blur(img_smooth_base, (ksize, ksize))
    axes[idx+1].imshow(box_filtered, cmap='gray')
    axes[idx+1].set_title(f"Box Filter ({ksize}x{ksize})")
    axes[idx+1].axis('off')

plt.suptitle("Box Averaging Filter at Increasing Kernel Sizes", fontsize=14)
plt.tight_layout()
plt.show()

# 2. Gaussian Filtering Across 5 Standard Deviations
sigmas = [1.0, 2.0, 4.0, 8.0, 16.0]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))

for idx, sig in enumerate(sigmas):
    gauss_filtered = ndimage.gaussian_filter(img_smooth_base, sigma=sig)
    axes[idx].imshow(gauss_filtered, cmap='gray')
    axes[idx].set_title(f"Gaussian (sigma = {sig})")
    axes[idx].axis('off')

plt.suptitle("Gaussian Low-Pass Filtering", fontsize=14)
plt.tight_layout()
plt.show()

# 3. Additive Gaussian Noise & Restoration
np.random.seed(42)
noisy_g1 = np.clip(img_smooth_base.astype(np.float32) + np.random.normal(0, 10, img_smooth_base.shape), 0, 255).astype(np.uint8)
noisy_g2 = np.clip(img_smooth_base.astype(np.float32) + np.random.normal(0, 20, img_smooth_base.shape), 0, 255).astype(np.uint8)
noisy_g3 = np.clip(img_smooth_base.astype(np.float32) + np.random.normal(0, 30, img_smooth_base.shape), 0, 255).astype(np.uint8)

restored_g1 = cv2.GaussianBlur(noisy_g1, (5, 5), 1.5)
restored_g2 = cv2.GaussianBlur(noisy_g2, (5, 5), 1.5)
restored_g3 = cv2.GaussianBlur(noisy_g3, (5, 5), 1.5)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(noisy_g1, cmap='gray'); axes[0, 0].set_title("Noisy Gaussian (sigma_n = 10)"); axes[0, 0].axis('off')
axes[0, 1].imshow(noisy_g2, cmap='gray'); axes[0, 1].set_title("Noisy Gaussian (sigma_n = 20)"); axes[0, 1].axis('off')
axes[0, 2].imshow(noisy_g3, cmap='gray'); axes[0, 2].set_title("Noisy Gaussian (sigma_n = 30)"); axes[0, 2].axis('off')

axes[1, 0].imshow(restored_g1, cmap='gray'); axes[1, 0].set_title("Restored (Gaussian Low-Pass)"); axes[1, 0].axis('off')
axes[1, 1].imshow(restored_g2, cmap='gray'); axes[1, 1].set_title("Restored (Gaussian Low-Pass)"); axes[1, 1].axis('off')
axes[1, 2].imshow(restored_g3, cmap='gray'); axes[1, 2].set_title("Restored (Gaussian Low-Pass)"); axes[1, 2].axis('off')

plt.suptitle("Additive Gaussian Noise Reduction via Gaussian Low-Pass Filtering", fontsize=14)
plt.tight_layout()
plt.show()

# 4. Salt-and-Pepper Noise & Median Filtering
def add_salt_pepper(img, prob):
    noisy = img.copy()
    num_salt = np.ceil(prob * img.size * 0.5)
    num_pepper = np.ceil(prob * img.size * 0.5)
    
    # Salt
    coords_s = [np.random.randint(0, i - 1, int(num_salt)) for i in img.shape]
    noisy[tuple(coords_s)] = 255
    # Pepper
    coords_p = [np.random.randint(0, i - 1, int(num_pepper)) for i in img.shape]
    noisy[tuple(coords_p)] = 0
    return noisy

sp_noisy1 = add_salt_pepper(img_smooth_base, 0.02)
sp_noisy2 = add_salt_pepper(img_smooth_base, 0.05)
sp_noisy3 = add_salt_pepper(img_smooth_base, 0.10)

med_rest1 = cv2.medianBlur(sp_noisy1, 3)
med_rest2 = cv2.medianBlur(sp_noisy2, 5)
med_rest3 = cv2.medianBlur(sp_noisy3, 7)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(sp_noisy1, cmap='gray'); axes[0, 0].set_title("S&P Noise (Density 2%)"); axes[0, 0].axis('off')
axes[0, 1].imshow(sp_noisy2, cmap='gray'); axes[0, 1].set_title("S&P Noise (Density 5%)"); axes[0, 1].axis('off')
axes[0, 2].imshow(sp_noisy3, cmap='gray'); axes[0, 2].set_title("S&P Noise (Density 10%)"); axes[0, 2].axis('off')

axes[1, 0].imshow(med_rest1, cmap='gray'); axes[1, 0].set_title("Restored (Median 3x3)"); axes[1, 0].axis('off')
axes[1, 1].imshow(med_rest2, cmap='gray'); axes[1, 1].set_title("Restored (Median 5x5)"); axes[1, 1].axis('off')
axes[1, 2].imshow(med_rest3, cmap='gray'); axes[1, 2].set_title("Restored (Median 7x7)"); axes[1, 2].axis('off')

plt.suptitle("Salt-and-Pepper Impulse Noise Reduction via Median Filtering", fontsize=14)
plt.tight_layout()
plt.show()


---
## SECTION 10: High-Pass Filtering, Edge Detection, and Image Sharpening

### Theoretical Background
High-pass spatial filtering highlights high-frequency details (edges, boundaries, fine textures):

#### 1. First-Order Gradients (Sobel Operators)
$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \quad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$
Gradient magnitude: $|\nabla f| = \sqrt{G_x^2 + G_y^2} \approx |G_x| + |G_y|$.

#### 2. Isotropic Second-Order Derivative (Laplacian Operator)
$$\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2} \implies \mathbf{K}_{\text{Lap}} = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$$

#### 3. Unsharp Masking & High-Boost Filtering
$$g_{\text{mask}}(x, y) = f(x, y) - f_{\text{smooth}}(x, y)$$
$$f_{\text{hb}}(x, y) = f(x, y) + k \cdot g_{\text{mask}}(x, y)$$
where $k=1$ corresponds to standard unsharp masking, and $k > 1$ represents high-boost filtering.


In [ ]:
# Section 10: Code Implementation

# Load CT Thorax
img_sharp_base = img_ct.copy()

# 1. Sobel Gradient Operators
sobel_x = cv2.Sobel(img_sharp_base, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_sharp_base, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.hypot(sobel_x, sobel_y)
sobel_mag_norm = (sobel_mag / np.max(sobel_mag) * 255.0).astype(np.uint8)

# 2. Laplacian Operator
laplacian = cv2.Laplacian(img_sharp_base, cv2.CV_64F)
laplacian_sharpened = np.clip(img_sharp_base.astype(np.float64) - laplacian, 0, 255).astype(np.uint8)

# 3. High-Boost Filtering across k = [0.5, 1.2, 2.5]
blurred = cv2.GaussianBlur(img_sharp_base, (5, 5), 1.0)
mask_unsharp = img_sharp_base.astype(np.float64) - blurred.astype(np.float64)

k_vals = [0.5, 1.2, 2.5]
hb_images = []
for k in k_vals:
    hb_img = np.clip(img_sharp_base.astype(np.float64) + k * mask_unsharp, 0, 255).astype(np.uint8)
    hb_images.append(hb_img)

# Display Visual Comparisons
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(np.abs(sobel_x), cmap='gray'); axes[0, 0].set_title("Sobel G_x (Vertical Edges)"); axes[0, 0].axis('off')
axes[0, 1].imshow(np.abs(sobel_y), cmap='gray'); axes[0, 1].set_title("Sobel G_y (Horizontal Edges)"); axes[0, 1].axis('off')
axes[0, 2].imshow(sobel_mag_norm, cmap='gray'); axes[0, 2].set_title("Sobel Magnitude |grad f|"); axes[0, 2].axis('off')

axes[1, 0].imshow(hb_images[0], cmap='gray'); axes[1, 0].set_title(f"High-Boost (k = {k_vals[0]})"); axes[1, 0].axis('off')
axes[1, 1].imshow(hb_images[1], cmap='gray'); axes[1, 1].set_title(f"High-Boost (k = {k_vals[1]})"); axes[1, 1].axis('off')
axes[1, 2].imshow(hb_images[2], cmap='gray'); axes[1, 2].set_title(f"High-Boost (k = {k_vals[2]})"); axes[1, 2].axis('off')

plt.suptitle("High-Pass Spatial Filtering, Edge Detection & High-Boost Sharpening", fontsize=14)
plt.tight_layout()
plt.show()


---
## SECTION 11: Advanced Spatial Enhancement Methods

### Theoretical Background
1. **Brightness-Preserving Bi-Histogram Equalization (BBHE)**: Splits histogram into two sub-histograms based on input mean intensity $I_m$, equalizing them independently to preserve original brightness.
2. **Contrast Limited Adaptive Histogram Equalization (CLAHE)**: Prevents noise over-amplification in LHE by clipping local histograms at a threshold limit $C_L$.
3. **Frangi Multiscale Vesselness Filter**: Analyzes eigenvalues $\lambda_1, \lambda_2$ of the 2D Hessian matrix $H(I)$ to enhance tubular vessel structures.
4. **Fuzzy Logic-Based Intensity Enhancement**: Transforms pixel intensities into fuzzy set membership functions, modifies fuzzy contrast, and defuzzifies.
5. **Total Variation (TV) Denoising**: Minimizes $E(u) = \int |\nabla u| dx + \frac{\lambda}{2} \|u - f\|^2$, removing noise while preserving sharp step edges.
6. **Perona–Malik Anisotropic Diffusion Filtering**: Non-linear partial differential equation smoothing:
   $$\frac{\partial I}{\partial t} = \text{div}\big(c(\|\nabla I\|) \nabla I\big), \quad c(\|\nabla I\|) = \exp\left(-\Big(\frac{\|\nabla I\|}{K}\Big)^2\right)$$


In [ ]:
# Section 11: Code Implementation

img_adv_base = img_ct.copy()

# 1. Brightness-Preserving Bi-Histogram Equalization (BBHE)
def bbhe(img):
    mean_val = int(np.mean(img))
    lower_mask = img <= mean_val
    upper_mask = img > mean_val
    
    img_lower = img[lower_mask]
    img_upper = img[upper_mask]
    
    hist_l, _ = np.histogram(img_lower, bins=mean_val+1, range=(0, mean_val+1))
    hist_u, _ = np.histogram(img_upper, bins=255-mean_val, range=(mean_val+1, 256))
    
    cdf_l = np.cumsum(hist_l) / hist_l.sum() if hist_l.sum() > 0 else np.zeros_like(hist_l)
    cdf_u = np.cumsum(hist_u) / hist_u.sum() if hist_u.sum() > 0 else np.zeros_like(hist_u)
    
    map_l = (mean_val * cdf_l).astype(np.uint8)
    map_u = (mean_val + 1 + (255 - mean_val - 1) * cdf_u).astype(np.uint8)
    
    out = np.zeros_like(img)
    out[lower_mask] = map_l[img[lower_mask]]
    out[upper_mask] = map_u[img[upper_mask] - (mean_val + 1)]
    return out

img_bbhe = bbhe(img_adv_base)

# 2. CLAHE
clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe_obj.apply(img_adv_base)

# 3. Frangi Vesselness Filter (with custom fallback)
def frangi_custom(image):
    float_img = image.astype(np.float32)
    Ixx = cv2.Sobel(cv2.Sobel(float_img, cv2.CV_32F, 1, 0, ksize=3), cv2.CV_32F, 1, 0, ksize=3)
    Iyy = cv2.Sobel(cv2.Sobel(float_img, cv2.CV_32F, 0, 1, ksize=3), cv2.CV_32F, 0, 1, ksize=3)
    Ixy = cv2.Sobel(cv2.Sobel(float_img, cv2.CV_32F, 1, 0, ksize=3), cv2.CV_32F, 0, 1, ksize=3)
    tmp = np.sqrt((Ixx - Iyy)**2 + 4 * Ixy**2)
    lambda1 = 0.5 * (Ixx + Iyy + tmp)
    lambda2 = 0.5 * (Ixx + Iyy - tmp)
    l1 = np.where(np.abs(lambda1) < np.abs(lambda2), lambda1, lambda2)
    l2 = np.where(np.abs(lambda1) < np.abs(lambda2), lambda2, lambda1)
    Rb = (l1 / (l2 + 1e-5))**2
    S2 = l1**2 + l2**2
    vesselness = np.exp(-Rb / 0.5) * (1.0 - np.exp(-S2 / 2.0))
    vesselness[l2 > 0] = 0
    return vesselness

try:
    img_frangi = skimage.filters.frangi(img_adv_base.astype(np.float32) / 255.0)
except Exception:
    img_frangi = frangi_custom(img_adv_base)

img_frangi_norm = (img_frangi / (np.max(img_frangi) + 1e-5) * 255.0).astype(np.uint8)

# 4. Fuzzy Logic Contrast Enhancement
def fuzzy_enhancement(img):
    r = img.astype(np.float32) / 255.0
    s = np.where(r <= 0.5, 2.0 * r**2, 1.0 - 2.0 * (1.0 - r)**2)
    return (s * 255.0).clip(0, 255).astype(np.uint8)

img_fuzzy = fuzzy_enhancement(img_adv_base)

# 5. Total Variation (TV) Denoising (with custom fallback)
def tv_denoise_custom(image, weight=0.1, n_iter=20):
    u = image.copy().astype(np.float32)
    for _ in range(n_iter):
        ux = np.roll(u, -1, axis=1) - u
        uy = np.roll(u, -1, axis=0) - u
        norm = np.sqrt(ux**2 + uy**2 + 1e-5)
        div = (ux / norm) - np.roll(ux / norm, 1, axis=1) + (uy / norm) - np.roll(uy / norm, 1, axis=0)
        u += 0.1 * (image - u + weight * div)
    return np.clip(u, 0, 1)

try:
    img_tv = skimage.restoration.denoise_tv_chambolle(img_adv_base.astype(np.float32) / 255.0, weight=0.1)
except Exception:
    img_tv = tv_denoise_custom(img_adv_base.astype(np.float32) / 255.0, weight=0.1)

img_tv_norm = (img_tv * 255.0).clip(0, 255).astype(np.uint8)

# 6. Perona-Malik Anisotropic Diffusion Filter
def anisodiff(img, niter=10, kappa=20, gamma=0.1):
    im = img.astype(np.float32)
    for _ in range(niter):
        dn = np.pad(im, ((0,1),(0,0)), mode='edge')[1:,:] - im
        ds = np.pad(im, ((1,0),(0,0)), mode='edge')[:-1,:] - im
        de = np.pad(im, ((0,0),(0,1)), mode='edge')[:,1:] - im
        dw = np.pad(im, ((0,0),(1,0)), mode='edge')[:,:-1] - im
        
        cn = np.exp(-(dn/kappa)**2)
        cs = np.exp(-(ds/kappa)**2)
        ce = np.exp(-(de/kappa)**2)
        cw = np.exp(-(dw/kappa)**2)
        
        im += gamma * (cn*dn + cs*ds + ce*de + cw*dw)
    return np.clip(im, 0, 255).astype(np.uint8)

img_aniso = anisodiff(img_adv_base, niter=15, kappa=25, gamma=0.1)

# Visualize All 6 Advanced Algorithms in 2x3 Subplot Grid
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(img_bbhe, cmap='gray'); axes[0, 0].set_title("1. BBHE (Bi-Histogram Eq)"); axes[0, 0].axis('off')
axes[0, 1].imshow(img_clahe, cmap='gray'); axes[0, 1].set_title("2. CLAHE (Adaptive Block Eq)"); axes[0, 1].axis('off')
axes[0, 2].imshow(img_frangi_norm, cmap='hot'); axes[0, 2].set_title("3. Frangi Vesselness Filter"); axes[0, 2].axis('off')

axes[1, 0].imshow(img_fuzzy, cmap='gray'); axes[1, 0].set_title("4. Fuzzy Logic Contrast"); axes[1, 0].axis('off')
axes[1, 1].imshow(img_tv_norm, cmap='gray'); axes[1, 1].set_title("5. Total Variation (TV) Denoising"); axes[1, 1].axis('off')
axes[1, 2].imshow(img_aniso, cmap='gray'); axes[1, 2].set_title("6. Perona-Malik Anisotropic Diffusion"); axes[1, 2].axis('off')

plt.suptitle("Advanced Spatial Domain Enhancement Algorithms", fontsize=14)
plt.tight_layout()
plt.show()
